In [3]:
import json 
from pathlib import Path
import tiktoken
import pandas as pd
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings

In [2]:
## intialize output directory
output_dir = Path("../data/chunked")

In [6]:
## embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en"
)


## initialize text splitter
text_splitter = SemanticChunker(embeddings=embedding_model)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3983.08it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
## initialize semantic chunking using SemanticChunker
# chunk_size = 512  # retained for reference (not used by SemanticChunker)
# overlap = 0

def chunk_text(text):
    # SemanticChunker splits based on semantic similarity (embedding space), not raw characters.
    return text_splitter.split_text(text)


In [8]:
## Load combined data
with open(r"..\data\combined_10k_data.json", "r") as f:
    data = json.load(f)

In [9]:
# Create chunks for each section and store in a list
all_chunks = []

print("Creating chunks...")
for company in data:
    file_name = company.get("file_name", "Unknown")
    # Extract company name from file_name (e.g., "aapl-20230930" -> "aapl")
    company_name = file_name.split("-")[0].upper() if file_name else "Unknown"
    
    # Iterate through sections
    for section in ["risk_factors", "management_discussion_and_analysis"]:
        text = company.get(section, "")
        
        if text:  # Only process if section exists and has content
            chunks = chunk_text(text)
            
            # Create chunk objects with metadata
            for chunk_id, chunk_content in enumerate(chunks):
                chunk_obj = {
                    "company_name": company_name,
                    "section": section,
                    "chunk_id": chunk_id,
                    "text": chunk_content
                }
                all_chunks.append(chunk_obj)

print(f"Total chunks created: {len(all_chunks)}\n")

Creating chunks...
Total chunks created: 447



In [12]:
print(all_chunks[0]["text"])
print(all_chunks[1]["text"])

The Companys business, reputation, results of operations, financial condition and stock price can be affected by a number of factors, whether currently known or unknown, including those described below. When any one or more of these risks materialize from time to time, the Companys business, reputation, results of operations, financial condition and stock price can be materially and adversely affected. Because of the following factors, as well as other factors affecting the Companys results of operations and financial condition, past financial performance should not be considered to be a reliable indicator of future performance, and investors should not use historical trends to anticipate results or trends in future periods. This discussion of risk factors contains forward-looking statements. This section should be read in conjunction with Part II, Item 7, Managements Discussion and Analysis of Financial Condition and Results of Operations and the consolidated financial statements and 

In [13]:
with open(str(output_dir) + "/chunks_type2_Semantic.json", "w") as f:
    json.dump(all_chunks, f, indent=2)

In [14]:
# chunk statistics
chunks_df = pd.DataFrame(all_chunks)

print("\n" + "="*50)
print("CHUNKS SUMMARY")
print("="*50)
print(f"Total chunks: {len(chunks_df)}")

print(f"\nChunks per company:")
print(chunks_df.groupby('company_name').size())

print(f"\nChunks per section:")
print(chunks_df.groupby('section').size())

print(f"\nChunks per company and section:")
print(chunks_df.groupby(['company_name', 'section']).size())

print(f"\n" + "="*50)
print("SAMPLE CHUNKS")
print("="*50)
for company in chunks_df['company_name'].unique()[:2]:
    for section in chunks_df['section'].unique()[:1]:
        sample = chunks_df[(chunks_df['company_name'] == company) & (chunks_df['section'] == section)].iloc[0]
        print(f"\n{company} - {section} (chunk {sample['chunk_id']}):")
        print(f"Text preview: {sample['text'][:200]}...")



CHUNKS SUMMARY
Total chunks: 447

Chunks per company:
company_name
AAPL          24
COCOCOLA     109
GOOG          25
LLY           35
MICROSOFT     40
NFLX          28
NVDA          40
ORACLE        71
UBER          75
dtype: int64

Chunks per section:
section
management_discussion_and_analysis    220
risk_factors                          227
dtype: int64

Chunks per company and section:
company_name  section                           
AAPL          management_discussion_and_analysis     6
              risk_factors                          18
COCOCOLA      management_discussion_and_analysis    75
              risk_factors                          34
GOOG          management_discussion_and_analysis     6
              risk_factors                          19
LLY           management_discussion_and_analysis    22
              risk_factors                          13
MICROSOFT     management_discussion_and_analysis    17
              risk_factors                          23
NFLX    